## COMP661 Assignment 3 Canterbury Temperature forecasting 1174130 Ken Livingston

**Part 5: Summary and Reflections**
This notebook draws the evidence from Parts 1–4 into one discussion. It does not repeat the detailed tables and figures - references such as *(Table 3.13)* or *(Fig. 2.3)* point to where the supporting evidence can be found:

| Part | Notebook | Content |
|---|---|---|
| 1 | `Part_1/src/Preprocessing.ipynb` | Data selection, cleaning, gap filling, correlation analysis |
| 2 | `Part_2/src/RNN_Model.ipynb` | Single-station RNN trained at Christchurch Aero, tested at all five stations |
| 3 | `Part_3/src/GNN_RNN_Model.ipynb` | Distance-weighted GNN + RNN forecasting all five stations together |
| 4 | `Part_4/src/Sensitivity_XAI.ipynb` | Feature/station removal, perturbation sensitivity, XAI |

**The central finding:**  Both models produce forecasts that are statistically sound and better than naive persistence (what we get tomorrow is what we have today). However, beyond one or two days they add almost nothing to the seasonal average, and they don't anticipate the large day-to-day changes (2 to 5°C or more) that matter most for decision-makers. The limiting factor is the information in the inputs, not the model architecture: eight daily variables from five closely spaced stations contain little advance warning of the weather systems that cause large changes. Most of the discussion below explains this conclusion, shows the evidence behind it, and considers what it means for operational use.


**5.1 Summary of Findings**

**The five stations**

| Station | Character | Role in the findings |
|---|---|---|
| **Christchurch Aero** (4843) | Central reference site; longest and most complete record (99.9% observed in the study period) | Part 2 source station; the station used for XAI |
| **Chch Kyle Street** (24120) | Urban site 7km from Christchurch Aero; warmest and least variable | Near-duplicate of Christchurch Aero (r = 0.98); its evaporation record ended on 4 Oct 2025, which set the end of the study period |
| **Rangiora** (17244) | ~20km north of Christchurch; similar coastal-plain climate | Closely tied to the Christchurch pair (r = 0.97–0.98) |
| **Ashburton Aero** (26170) | ~75 km south-west; oldest record (Feb 2006) | Set the start of the study period; the only station to the south-west |
| **Hanmer Forest** (11234) | ~110 km north, inland, at higher altitude; coldest nights, largest temperature range, lightest wind, most rain | The climatic outlier (r = 0.92–0.93 with the others); most gaps (393-day evaporation outage); hardest to forecast |

The network is therefore three broadly interchangeable stations around Christchurch, plus two outliers. This structure explains most of the spatial results below.

**Data-preparation decisions that shaped the analysis**
* *Common period 1 Mar 2006 – 30 Sep 2025* The start was set by Ashburton Aero and the end by Chch Kyle Street's evaporation record. This gave ~19.6 years and about 20 seasonal cycles. Older data (Christchurch Aero from 1953) could not be used under the all-stations rule.
* *Date alignment* All readings are 9 am NZ time stored in UTC. Converting to local dates before merging avoided any possible misalignment between files, which would have quietly damaged every later step.
* *Tiered gap filling* Gaps were filled by derivation (mean = (max + min)/2), interpolation for ≤ 3 days, and regression from the most correlated neighbouring station for longer gaps and all rainfall. Only 1.6% of values were missing, but Hanmer Forest's evaporation is 14% imputed. This is a known weakness at the station that was already the hardest.
* *Chronological split and training-only scaling* Datasets were broadly: Train 2006–2019, Validation 2019–2022, Test 2022–2025. The test years turned out to be about 0.4°C warmer than the training years (Table 1.18), a realistic outcome for a model facing a warming climate.
* *Per-station scaling (Part 3)* This proved to be the most consequential modelling decision of the whole project (see below).
* *Fixed inputs.* The brief specified exactly eight features, so no calendar (day-of-year) inputs were added. The model had to infer the season from the data, and it learnt to do this mainly from evaporation.

**How predictable is 7-day mean temperature at these stations?**
Part 1 set the ceiling before any model was trained. Almost all of the correlation between today's inputs and temperature a week ahead is seasonal (raw r ≈ 0.6). Once the seasonal cycle is removed, today's temperature anomaly predicts tomorrow's (r ≈ 0.45), but the relationship fades to insignificance by t+3 to t+4 (Fig. 1.5). Every model result that followed matches this ceiling.

**RNN vs GNN + RNN**

| | RNN (Part 2) | GNN + RNN (Part 3) |
|---|---|---|
| Training data | Christchurch Aero only | All five stations, each in its own scaled units |
| Test RMSE, mean of 5 stations | 3.14°C | 3.08°C |
| Best / worst station | Kyle St 2.85 / Hanmer 3.72°C | Kyle St 2.88 / Hanmer 3.46°C |
| Largest bias | +1.31°C (Hanmer; +2.06°C in winter) | +0.34°C (Kyle St) |
| Skill vs each station's climatology | −4% to +2% | +1% to +3% at every station |

* *Both models* beat persistence comfortably (by ~1°C RMSE), but they beat the seasonal average only slightly. Their advantage is concentrated at t+1 (about 15% lower error than climatology), shrinks at t+2, and has gone from t+3 onward, where both effectively forecast the seasonal average (Figs 2.3, 3.4).
* *The GNN + RNN is the better all-round model* because it removes the Part 2 model's main failure, which was that the single-station RNN, trained on Christchurch's climate, carried a large warm bias to inland Hanmer Forest. The GNN + RNN cuts Hanmer's error by 7% and its bias from +1.31 to +0.21°C. At the two Christchurch stations it is very slightly worse (+0.02–0.03°C), where the Part 2 specialist had the advantage.

**Did spatial information and distance-based edge weights help?**
No, not measurably. This is one of the clearest results of the investigation:
* The no-graph ablation (identical model, edges removed) matched the GNN + RNN on the test set (mean RMSE 3.076 vs 3.084°C, Table 3.13). All of the improvement over Part 2 came from training one model on all stations with per-station scaling, not from the graph.
* The edge-weight scale made no difference (σ = 77.8 km vs 48.3 km), and the processing order (temporal then spatial versus spatial then temporal) and the number of graph layers changed validation error by ≤ 0.01°C (Table 3.8).
* Removing any single station and retraining changed accuracy elsewhere by at most ±0.015°C, which was well within the noise between random seeds (Table 4.3).

The XAI results show the model does use its neighbours. 45–62% of the attribution for a Christchurch Aero forecast comes from other stations, mainly Kyle Street and Rangiora (Table 4.8). But it also showed that information is redundant - it duplicates what the station already knows. There are two physical reasons:
1. *The stations are too close and too similar.* At 7–23 km apart, the Christchurch stations see the same weather on the same day.
2. *The time step is too coarse.* Canterbury weather systems such as a southerly change or a nor'wester cross 7–174 km in hours. At a daily resolution weather arrives at all five stations on the same day, so no station gives another any warning. A change seen at Ashburton reaches Rangiora roughly two hours later, which is invisible in daily data.

**Sensitivity analysis and XAI**
* *Evaporation is the model's seasonal clock* The two PET features (Priestley-Taylor and Penman) are the most influential inputs overall, together with mean temperature (Table 4.9). Removing all evaporation features is the only change that clearly hurts (+0.15°C, mostly at t+7), and the t+7 forecast is most sensitive to PET (≈ 1.5 to 1.7°C per σ). PET follows sunshine, which identifies the time of year more reliably than temperature does - spring and autumn can share temperatures but not sunshine.
* *Recent temperatures and rain drive the 1-day forecast* At t+1 the forecast is most sensitive to mean and minimum temperature. Rainfall, almost uncorrelated with temperature a week ahead, matters for tomorrow (+0.09°C error when removed). XAI shows it is the dominant reason the model correctly forecast a cold day after heavy rain.
* *Wind speed is used but not needed* The model responds to wind (−0.75°C per σ), yet retraining without it costs nothing. Its information is also contained in other inputs, especially Penman open-water evaporation (https://www.sciencedirect.com/science/article/abs/pii/S002216940600326X).
* *Each station's own sensors are critical* Losing them at forecast time raises that station's error by 1.1 to 1.4°C, which makes it worse than climatology. The neighbours do not compensate (Table 4.4).
* *The methods disagree in instructive ways* Correlation and global importance under-rate rainfall's short-lived, event-driven value; attribution over-states the neighbours' necessity; and permutation importance under-rates correlated inputs. Using several methods was essential to avoid wrong conclusions.

**Where the models performed well and poorly**
* *Good:* Forecasts one day ahead in settled or slowly changing conditions, and persistent anomalies. Examples are the warm spell of early March 2025 (Case 1, within 1°C) and the cold, wet change of late March 2023 (Case 2, within 0.8°C). In both, the signal was already in the previous day's data - a warm trend for Case 1, or 32.8mm of rain and a sharp drop in temperature for Case 2.
* *Poor:* Sudden changes with no precursor in the inputs. The worst forecast was a (probable) nor'wester on 20 Jan 2024 (Case 3) - mean temperature rose 8.6°C overnight to 28.1°C and was under-forecast by almost 10°C. XAI showed the model had correctly recognised an ordinary warm summer spell, but nothing in its inputs announced the spike.
* *Systematically:* forecasts are conservative. They rarely go below about 5°C or above about 20°C while observations range from 0 to 28°C (Fig. 2.6). The 7-day forecast collapses to the seasonal curve (Fig. 2.5). This is the expected behaviour of a model minimising squared error when the outcome is uncertain - it hedges towards the average.

**Interpretation for decision-making**
For practical purposes, the forecasts are not meaningfully better than the seasonal average. The skill over climatology is 1–3% across the week, and RMSE is about 3°C. Day-to-day changes in mean temperature are themselves about 3°C (persistence error at t+1 = 3.2°C, Fig. 2.3). The model's typical error is therefore the same size as the 2 to 5°C or more changes that decision-makers need to know about, and the model has a structural tendency to smooth exactly those changes away. The models find small, statistically real improvements, but they cannot provide the early warning of large changes that would make a 7-day forecast operationally valuable. As the XAI and spatial results show, that warning information is not present in the inputs.

**Recommendations that might improve the model**
The main recommendations are in the input data:
1. Additional inputs, such as wind direction, sunshine and pressure, might give more useful clues about temperatures beyond t+2 or t+3.
2. Data from stations further afield, such as Invercargill or Dunedin, and Greymouth or Westport, might give some advance warning of weather changes for the following day(s).